# Tie-break ablation (A2)

Rank the same scores under three operators and measure the disagreement. Because
all three consume **bit-identical** scores, any disagreement is caused by the
tie-break and by nothing else.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
from tfidf_stability.analysis.tie_break_ablations import ablate_queries, disagreement_rate
from tfidf_stability.datasets.loaders import load_dataset
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.ranking.attributes import AttributeTable
from tfidf_stability.ranking.ranker import rank_all_operators
from tfidf_stability.similarity.cosine import cosine_against_corpus
from tfidf_stability.utils.numerics import same_bits
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

data = load_dataset("synthetic_tiny")
pipeline = PreprocessingPipeline()
features = [pipeline.preprocess(str(r["text"])) for r in data.records]
model = TfidfVectoriser().fit(features, data.doc_ids)
table = AttributeTable.from_records(data.records)
documents = [model.document(i) for i in range(model.n_documents)]

scores_by_query = [
    cosine_against_corpus(
        TfidfVectoriser.transform_query(list(f)[:6], model), documents, model.norms)
    for f in features[::4]
]

# A2's premise, checked rather than assumed.
premise_holds = True
for scores in scores_by_query:
    rankings = rank_all_operators(scores, table)
    reference = rankings["pi"].sorted_scores
    for ranking in rankings.values():
        if not all(same_bits(a, b) for a, b in zip(ranking.sorted_scores, reference)):
            premise_holds = False
print("all operators saw bit-identical scores:", premise_holds)

In [ ]:
ks = tuple(k for k in (1, 5, 10, 20, 50) if k < model.n_documents)
results = ablate_queries([(f"q{i}", s) for i, s in enumerate(scores_by_query)], table, ks=ks)

pairs = sorted({(p.baseline, p.variant) for r in results for p in r.pairs})
for baseline, variant in pairs:
    cells = []
    for k in ks:
        rate, n = disagreement_rate(results, baseline, variant, k)
        cells.append(f"k{k}={rate:5.1%}(n={n})")
    print(f"{baseline:9} vs {variant:9}  " + "  ".join(cells))

Disagreement concentrates at **k=1**: the top document frequently sits in an
exact-tie block, so which one is returned first is decided entirely by the
tie-break. That is the decision-level discontinuity A2 names, at the rank where a
recommender's output is most visible.

Every rate carries its denominator. A rate without its `n` cannot be
distinguished from noise over three queries.